# 最小 GRPO 实验 · 用 trl 训一个「答案格式遵循」的小模型

**目标**：在小模型（默认 `Qwen/Qwen2.5-0.5B-Instruct`）上跑一个最小 GRPO 实验，让模型学会输出固定格式的算术题答案：

```
<think> ... 任意推理 ... </think>
<answer>NUMBER</answer>
```

Reward 完全可验证：
- `+1.0`：`<answer>` 中数值与正确答案一致。
- `+0.2`：格式正确但数值错。
- `0`：格式不对。

这是 *RLVR + GRPO* 的最简化版，跑通后你会对 9 章笔记里的算法有非常直观的体会。

> 资源建议：1 张 8GB 显存 GPU 即可（Qwen2.5-0.5B + bsz 1 + grad accum）。
> 如果没有 GPU，可以注释掉 `trainer.train()` 仅做 dry-run 看 loss 公式。

In [ ]:
# 依赖（trl >= 0.11 支持 GRPOTrainer）
# pip install -U "trl>=0.11.0" "transformers>=4.46" "datasets" "accelerate" "peft"

In [ ]:
import re, random
from datasets import Dataset

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

PROMPT_TEMPLATE = (
    '请严格按以下格式作答：\n'
    '<think>逐步推理</think>\n'
    '<answer>仅一个数值，不带单位</answer>\n\n'
    '题目：{q}'
)

def gen_arith():
    a = random.randint(1, 50); b = random.randint(1, 50)
    op = random.choice(['+', '-', '*'])
    expr = f'{a}{op}{b}'
    return {'q': f'{a} {op} {b} = ?', 'gold': str(eval(expr))}

random.seed(0)
DATA = [gen_arith() for _ in range(512)]
ds = Dataset.from_list(DATA)
ds = ds.map(lambda x: {'prompt': PROMPT_TEMPLATE.format(q=x['q']), 'gold': x['gold']})
ds[:3]

## 1. 定义 reward 函数（可验证奖励）

TRL 的 `GRPOTrainer` 接受 `reward_funcs: list[Callable]`，每个函数签名 `f(prompts, completions, **kw) -> list[float]`。

In [ ]:
ANS_RE = re.compile(r'<answer>\s*(-?\d+(?:\.\d+)?)\s*</answer>')
FORMAT_RE = re.compile(r'<think>.*?</think>\s*<answer>.*?</answer>', re.S)

def reward_correct(completions, gold, **kw):
    rewards = []
    for comp, g in zip(completions, gold):
        m = ANS_RE.search(comp)
        if not m:
            rewards.append(0.0)
        elif m.group(1).strip() == g:
            rewards.append(1.0)
        else:
            rewards.append(0.2)
    return rewards

def reward_format(completions, **kw):
    return [0.1 if FORMAT_RE.search(c) else 0.0 for c in completions]

# 假演示，用一个虚拟回答验证 reward 函数行为
test_completions = [
    '<think>1+2=3</think>\n<answer>3</answer>',
    '<think>?</think>\n<answer>9</answer>',
    '答案是 3。',
]
print('correct:', reward_correct(test_completions, ['3', '3', '3']))
print('format :', reward_format(test_completions))

## 2. GRPOTrainer 配置

关键超参：
- `num_generations`（GRPO 中的 G，组内采样数）
- `beta`（KL 系数）
- `learning_rate` 通常很小（1e-6 到 5e-6）

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype='auto')

cfg = GRPOConfig(
    output_dir='runs/grpo_arith',
    learning_rate=2e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,        # GRPO 的 G
    max_prompt_length=128,
    max_completion_length=128,
    beta=0.01,                # KL 锚
    num_train_epochs=1,
    logging_steps=10,
    save_strategy='no',
    report_to=[],
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_correct, reward_format],
    args=cfg,
    train_dataset=ds,
)
# 真实训练（需 GPU 资源）：
# trainer.train()

## 3. 训练前/后对比（如果跑了 train）

训练前 base model 经常忽略格式；训练后会稳定输出 `<think>...</think><answer>...</answer>`。

In [ ]:
import torch

def gen_one(model, tokenizer, q):
    p = PROMPT_TEMPLATE.format(q=q)
    inp = tokenizer(p, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

for q in ['12 + 7 = ?', '23 - 9 = ?', '6 * 8 = ?']:
    print('Q:', q)
    print('A:', gen_one(model, tokenizer, q))
    print('---')

## 4. 思考

- *Reward 设计*：「正确答案」 + 「格式」组合是经典 RLVR 套路；多 reward 加权可让模型先学格式再学正确。
- *KL 锚 β*：太大会限制模型学习；太小会把 base 能力毁掉。
- *组大小 G*：G=8 是常用甜点；越大方差越小但越贵。
- *扩展*：把 `gen_arith` 换成 GSM8K，模型从 0.5B 换到 7B，把 reward 加上 *long-CoT* 鼓励项，就接近论文复现。

## 进阶练习

1. 加 DAPO 的 *clip-higher* 与 *dynamic sampling*（trl 0.13+ 已支持）。
2. 把 reward 换成「调用计算器工具的正确性」：模型生成代码 → 执行 → 判断结果。这一步就接近 *agentic RL*。
3. 用 vLLM 加速 rollout，把 num_generations 提到 32+，观察收敛曲线。